# FingerBall v2 — Stage 2A：Pose誘導による投球手追跡

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atstuyuki/FingerBall-Analysis/blob/main/notebooks/FingerBall_v2_Stage2A_Pose_Flow_WiLoR.ipynb)

MediaPipe Poseで肩・肘・手首を推定し、YOLO/OpenVINO検出候補を投球手首付近に制限。検出欠損ではLK光学フローとPose ROIで補完し、WiLoR INT8をCPUで実行します。ROI補完は正解検出ではありません。モデル・研究動画はGitHubに置かないでください。

## 0. 環境セットアップ

In [ ]:
%pip -q install "openvino>=2025,<2027" "mediapipe==0.10.35" "opencv-python-headless>=4.8" matplotlib pandas pyyaml tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. 設定

In [ ]:
from pathlib import Path
import os,time,json,shutil,urllib.request
import numpy as np,pandas as pd,cv2,openvino as ov,mediapipe as mp,yaml
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from google.colab import files
PROJECT_NAME='母指回内トラッキング'
PROJECT_PATH=''
VIDEO_PATH=''
THROWING_HAND='right'
YOLO_CLASS_RIGHT=1
CONF_MIN=0.10
POSE_VIS_MIN=0.35
WRIST_GATE=1.4
FLOW_MAX_AGE=12
FLOW_MIN_VALID=0.45
STRIDE=1
RUN_WILOR=True
CACHE=Path('/content/fingerball_stage2a_cache')
RESULTS_NAME='FingerBall_v2_stage2a_results'
POSE_TASK_URL='https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker/float16/1/pose_landmarker_full.task'
assert THROWING_HAND in ('right','left') and STRIDE>=1
print('OpenVINO:',ov.__version__,'MediaPipe:',mp.__version__)

## 2. Google Drive上の既存モデルをコピー

In [ ]:
def find_project():
    if PROJECT_PATH:
        p=Path(PROJECT_PATH)
        if not p.is_dir():raise FileNotFoundError(p)
        return p
    for root in [Path('/content/drive/MyDrive'),Path('/content/drive/Shareddrives')]:
        if (root/PROJECT_NAME).is_dir():return root/PROJECT_NAME
        if root.name=='Shareddrives' and root.is_dir():
            for d in root.iterdir():
                if (d/PROJECT_NAME).is_dir():return d/PROJECT_NAME
    raise FileNotFoundError('PROJECT_PATHを指定してください')
project=find_project()
a=project/'thumb_pronation/out/bench_light/verify/int8_leftcal'
b=project/'models/_DATA/wilor/pretrained_models/detector_openvino_model_288x512'
sources=[a/'wilor_core_int8_leftcal.xml',a/'wilor_core_int8_leftcal.bin',b/'detector.xml',b/'detector.bin',b/'metadata.yaml']
if not all(p.is_file() for p in sources):raise FileNotFoundError([(str(p),p.is_file()) for p in sources])
CACHE.mkdir(exist_ok=True,parents=True)
for src in sources:
    folder=CACHE/('wilor' if 'wilor_core' in src.name else 'detector')
    folder.mkdir(exist_ok=True)
    dest=folder/src.name
    if not dest.exists() or dest.stat().st_size!=src.stat().st_size:shutil.copy2(src,dest)
meta=yaml.safe_load((CACHE/'detector/metadata.yaml').read_text(encoding='utf8'))
print('Source:',project,'Hand classes:',meta.get('names'))

## 3. MediaPipe Pose Task / OpenVINOを準備

In [ ]:
task=CACHE/'pose_landmarker_full.task'
if not task.is_file():urllib.request.urlretrieve(POSE_TASK_URL,task)
assert task.stat().st_size>1000000
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
po=mp_vision.PoseLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=str(task)),
    running_mode=mp_vision.RunningMode.VIDEO,num_poses=1,
    min_pose_detection_confidence=.4,min_pose_presence_confidence=.4,min_tracking_confidence=.4)
pose=mp_vision.PoseLandmarker.create_from_options(po)
core=ov.Core()
det=core.compile_model(core.read_model(str(CACHE/'detector/detector.xml')),'CPU')
wilor=core.compile_model(core.read_model(str(CACHE/'wilor/wilor_core_int8_leftcal.xml')),'CPU')
dims=list(det.input(0).shape)
assert len(dims)==4 and dims[1]==3,dims
net_h,net_w=map(int,dims[2:])
nc=len(meta.get('names',{0:'left',1:'right'}))
print('Detector:',dims,'WiLoR inputs:',[list(p.shape) for p in wilor.inputs])
print('WiLoR outputs:',[list(p.shape) for p in wilor.outputs])

## 4. 動画を読み込む

In [ ]:
if not VIDEO_PATH:
    uploaded=files.upload()
    if not uploaded:raise RuntimeError('動画未選択')
    VIDEO_PATH=str(Path('/content')/next(iter(uploaded)))
video_src=Path(VIDEO_PATH)
if not video_src.is_file():raise FileNotFoundError(video_src)
source=video_src
if str(video_src).startswith('/content/drive/'):
    local=CACHE/('input_'+video_src.name)
    if not local.exists() or local.stat().st_size!=video_src.stat().st_size:shutil.copy2(video_src,local)
    source=local
cap=cv2.VideoCapture(str(source))
if not cap.isOpened():raise RuntimeError('OpenCVで動画を開けません')
total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT));fps=float(cap.get(cv2.CAP_PROP_FPS))
W=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH));H=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
ok,im=cap.read();cap.release()
if not ok:raise RuntimeError('動画が読めません')
print('Movie:',video_src.name,'frames:',total,'fps:',fps,'image:',W,H)
plt.figure(figsize=(10,5));plt.imshow(cv2.cvtColor(im,cv2.COLOR_BGR2RGB));plt.axis('off');plt.show()